# DichVideo Batch OmniVoice from SRT

Upload many `.srt` files and one reference voice audio. This notebook uses a high-quality OmniVoice profile, then merges all subtitle lines in each SRT into one text block and creates one full 24-bit WAV audio file with OmniVoice.

Use only with your own voice or with clear permission from the voice owner. Runtime > Change runtime type > GPU before running.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# OmniVoice README recommends torch/torchaudio 2.8.0 CUDA 12.8.
!pip -q install --force-reinstall --no-deps torch==2.8.0+cu128 torchaudio==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip -q install -U git+https://github.com/k2-fsa/OmniVoice.git faster-whisper soundfile


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

SRT_DIR = Path('/content/dichvideo_srt_uploads')
OUTPUT_DIR = Path('/content/dichvideo_omnivoice_audio')
shutil.rmtree(SRT_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
SRT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
REF_AUDIO = None
for name, data in uploaded.items():
    suffix = Path(name).suffix.lower()
    if suffix == '.srt':
        (SRT_DIR / name).write_bytes(data)
    elif suffix in {'.mp3', '.wav', '.m4a', '.flac', '.ogg'}:
        ref_path = Path('/content') / name
        ref_path.write_bytes(data)
        REF_AUDIO = str(ref_path)

if not REF_AUDIO:
    raise RuntimeError('Upload one reference audio file, e.g. audio-truyen.mp3')

print('Reference audio:', REF_AUDIO)
print('SRT files:')
for path in sorted(SRT_DIR.glob('*.srt')):
    print('-', path.name)


In [ ]:
import subprocess, torch
from faster_whisper import WhisperModel

REFERENCE_WAV = '/content/ref.wav'
REFERENCE_START = '0'
REFERENCE_DURATION = '20'
LANGUAGE_ID = 'vi'
ASR_MODEL = 'medium'

# Best-quality default: use about 20 seconds of clean speech, mono 24 kHz.
# If the source has leading silence, change REFERENCE_START to where speech begins.
subprocess.run([
    'ffmpeg', '-y', '-i', REF_AUDIO,
    '-ss', REFERENCE_START, '-t', REFERENCE_DURATION,
    '-ar', '24000', '-ac', '1',
    REFERENCE_WAV,
], check=True)

print('Reference WAV:', REFERENCE_WAV)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
asr = WhisperModel(ASR_MODEL, device=device, compute_type=compute_type)
segments, info = asr.transcribe(REFERENCE_WAV, language=LANGUAGE_ID, vad_filter=True)
REF_TEXT = ' '.join(seg.text.strip() for seg in segments).strip()
print('REF_TEXT =', REF_TEXT)
if not REF_TEXT:
    raise RuntimeError('Whisper did not recognize text from the reference audio. Use a clearer reference clip.')


In [ ]:
%%writefile /content/colab_batch_omnivoice_from_srt.py
from __future__ import annotations

import argparse
import json
import logging
import re
import shutil
import subprocess
import time
from pathlib import Path


def main() -> None:
    parser = argparse.ArgumentParser(description="Batch OmniVoice TTS from uploaded SRT files, merged to one audio per SRT.")
    parser.add_argument("--srt-dir", required=True)
    parser.add_argument("--output-dir", required=True)
    parser.add_argument("--ref-audio", required=True)
    parser.add_argument("--ref-text", default=None)
    parser.add_argument("--model", default="k2-fsa/OmniVoice")
    parser.add_argument("--device", default="cuda:0")
    parser.add_argument("--dtype", default="float16", choices=["float16", "float32"])
    parser.add_argument("--timing-mode", default="no_cut_sequential", choices=["fit_segments", "no_cut_sequential"])
    parser.add_argument("--max-tempo", type=float, default=1.35)
    parser.add_argument("--reference-start", type=float, default=0.0)
    parser.add_argument("--reference-duration", type=float, default=20.0)
    parser.add_argument("--speed", type=float, default=1.0)
    parser.add_argument("--num-step", type=int, default=32)
    args = parser.parse_args()

    srt_dir = Path(args.srt_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    logger = _setup_logger(output_dir / "batch_omnivoice_from_srt.log")
    _check_binary("ffmpeg")
    _check_binary("ffprobe")

    srt_files = sorted(srt_dir.glob("*.srt"))
    if not srt_files:
        raise RuntimeError(f"No .srt files found in {srt_dir}")

    ref_audio = _prepare_reference_audio(
        Path(args.ref_audio),
        output_dir / "reference_24k.wav",
        args.reference_start,
        args.reference_duration,
        logger,
    )

    import soundfile as sf
    import torch
    from omnivoice import OmniVoice

    dtype = torch.float16 if args.dtype == "float16" else torch.float32
    logger.info("Loading OmniVoice model=%s device=%s dtype=%s", args.model, args.device, args.dtype)
    model = OmniVoice.from_pretrained(args.model, device_map=args.device, dtype=dtype)

    for srt_index, srt_path in enumerate(srt_files, start=1):
        logger.info("Processing SRT %s/%s: %s", srt_index, len(srt_files), srt_path)
        segments = parse_srt(srt_path.read_text(encoding="utf-8-sig"))
        if not segments:
            logger.warning("Skipping empty SRT: %s", srt_path)
            continue
        _process_one_srt(
            model=model,
            sf=sf,
            srt_path=srt_path,
            segments=segments,
            ref_audio=ref_audio,
            ref_text=args.ref_text,
            output_dir=output_dir,
            timing_mode=args.timing_mode,
            max_tempo=args.max_tempo,
            speed=args.speed,
            num_step=args.num_step,
            logger=logger,
        )

    zip_base = output_dir.parent / "omnivoice_audio_results"
    if zip_base.with_suffix(".zip").exists():
        zip_base.with_suffix(".zip").unlink()
    shutil.make_archive(str(zip_base), "zip", output_dir)
    logger.info("Created zip: %s.zip", zip_base)


def _process_one_srt(model, sf, srt_path: Path, segments: list[dict], ref_audio: Path, ref_text: str | None, output_dir: Path, timing_mode: str, max_tempo: float, speed: float, num_step: int, logger: logging.Logger) -> None:
    name = _safe_stem(srt_path)
    per_srt_dir = output_dir / name
    per_srt_dir.mkdir(parents=True, exist_ok=True)

    merged_text = " ".join(" ".join(seg["text"].split()) for seg in segments).strip()
    if not merged_text:
        logger.warning("Skipping SRT without text: %s", srt_path)
        return

    raw_wav = per_srt_dir / f"{name}_raw.wav"
    full_wav = per_srt_dir / f"{name}_full.wav"
    logger.info("OmniVoice generate merged srt=%s segments=%s chars=%s", srt_path.name, len(segments), len(merged_text))
    audio = model.generate(
        text=merged_text,
        ref_audio=str(ref_audio),
        ref_text=ref_text,
        speed=speed,
        num_step=num_step,
    )
    sf.write(str(raw_wav), audio[0], 24000, subtype="PCM_24")
    _trim_silence(raw_wav, full_wav, logger)

    (per_srt_dir / f"{name}.srt").write_text(srt_path.read_text(encoding="utf-8-sig"), encoding="utf-8")
    (per_srt_dir / "merged_text.txt").write_text(merged_text, encoding="utf-8")
    duration = _duration(full_wav, logger)
    _write_json(per_srt_dir / "timing_schedule.json", [{
        "index": 1,
        "start": segments[0]["start"],
        "end": segments[-1]["end"],
        "text": merged_text,
        "source_segment_count": len(segments),
        "scheduled_start": 0.0,
        "scheduled_end": duration,
        "audio_duration": duration,
        "raw_audio": raw_wav.name,
        "mix_audio": full_wav.name,
    }])
    logger.info("Finished merged %s -> %s", srt_path.name, full_wav)


def _trim_silence(input_path: Path, output_path: Path, logger: logging.Logger) -> None:
    _run(["ffmpeg", "-y", "-i", str(input_path), "-af", "silenceremove=start_periods=1:start_duration=0.15:start_threshold=-45dB:stop_periods=1:stop_duration=0.35:stop_threshold=-45dB", "-ac", "1", "-ar", "44100", "-c:a", "pcm_s24le", str(output_path)], logger)

def parse_srt(content: str) -> list[dict]:
    content = content.replace("\r\n", "\n").replace("\r", "\n").strip()
    if not content:
        return []
    blocks = re.split(r"\n\s*\n", content)
    segments = []
    fallback_index = 1
    for block in blocks:
        lines = [line.strip() for line in block.split("\n") if line.strip()]
        if not lines:
            continue
        timing_line_index = next((i for i, line in enumerate(lines) if "-->" in line), None)
        if timing_line_index is None:
            continue
        maybe_index = lines[0] if timing_line_index > 0 else str(fallback_index)
        try:
            index = int(re.sub(r"\D+", "", maybe_index) or fallback_index)
        except ValueError:
            index = fallback_index
        timing = lines[timing_line_index]
        start_s, end_s = [part.strip().split()[0] for part in timing.split("-->", 1)]
        text = " ".join(lines[timing_line_index + 1:]).strip()
        if text:
            segments.append({
                "index": index,
                "start": _parse_srt_timestamp(start_s),
                "end": _parse_srt_timestamp(end_s),
                "text": text,
            })
            fallback_index += 1
    return segments


def _parse_srt_timestamp(value: str) -> float:
    match = re.match(r"(\d+):(\d+):(\d+)[,.](\d+)", value)
    if not match:
        raise ValueError(f"Invalid SRT timestamp: {value}")
    h, m, s, ms = match.groups()
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")[:3]) / 1000


def _mix_scheduled_audio(scheduled: list[dict], output_path: Path, logger: logging.Logger, root_dir: Path) -> None:
    total_duration = max(float(item["scheduled_end"]) for item in scheduled)
    silence_path = root_dir / "_silence.wav"
    _run(["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=channel_layout=mono:sample_rate=44100", "-t", f"{total_duration:.3f}", str(silence_path)], logger)
    inputs = ["-i", str(silence_path)]
    filters = []
    mix_inputs = ["[0:a]"]
    for input_index, item in enumerate(scheduled, start=1):
        audio_path = root_dir / item["mix_audio"]
        inputs.extend(["-i", str(audio_path)])
        delay_ms = max(0, int(float(item["scheduled_start"]) * 1000))
        label = f"a{input_index}"
        filters.append(f"[{input_index}:a]adelay={delay_ms}:all=1[{label}]")
        mix_inputs.append(f"[{label}]")
    filter_complex = ";".join(filters + [f"{''.join(mix_inputs)}amix=inputs={len(mix_inputs)}:normalize=0[out]"])
    _run(["ffmpeg", "-y", *inputs, "-filter_complex", filter_complex, "-map", "[out]", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s24le", str(output_path)], logger)
    silence_path.unlink(missing_ok=True)


def _prepare_reference_audio(input_path: Path, output_path: Path, start: float, duration: float, logger: logging.Logger) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    _run([
        "ffmpeg", "-y", "-i", str(input_path),
        "-ss", f"{start:.3f}", "-t", f"{duration:.3f}",
        "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s24le",
        str(output_path),
    ], logger)
    return output_path


def _fit_audio(input_path: Path, output_path: Path, target_duration: float, max_tempo: float, trim: bool, logger: logging.Logger) -> None:
    source_duration = _duration(input_path, logger)
    tempo = source_duration / target_duration if target_duration > 0 else 1.0
    audio_filter = f"atempo={min(tempo, max_tempo):.5f},apad" if tempo > 1.0 else "apad"
    if trim:
        audio_filter += f",atrim=0:{target_duration:.3f}"
    _run(["ffmpeg", "-y", "-i", str(input_path), "-filter:a", audio_filter, "-ac", "1", "-ar", "44100", "-c:a", "pcm_s24le", str(output_path)], logger)


def _tempo_audio(input_path: Path, output_path: Path, tempo: float, logger: logging.Logger) -> None:
    _run(["ffmpeg", "-y", "-i", str(input_path), "-filter:a", f"atempo={tempo:.5f}", "-ac", "1", "-ar", "44100", "-c:a", "pcm_s24le", str(output_path)], logger)


def _convert_audio(input_path: Path, output_path: Path, logger: logging.Logger) -> None:
    _run(["ffmpeg", "-y", "-i", str(input_path), "-ac", "1", "-ar", "44100", "-c:a", "pcm_s24le", str(output_path)], logger)


def _duration(path: Path, logger: logging.Logger) -> float:
    completed = _run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)], logger)
    return float(completed.stdout.strip())


def _safe_stem(path: Path) -> str:
    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", path.stem).strip("._")
    return stem or "srt"


def _run(cmd: list[str], logger: logging.Logger) -> subprocess.CompletedProcess:
    logger.info("Running command: %s", " ".join(cmd))
    completed = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if completed.stdout.strip():
        logger.info("stdout: %s", completed.stdout.strip()[-3000:])
    if completed.stderr.strip():
        logger.info("stderr: %s", completed.stderr.strip()[-3000:])
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with code {completed.returncode}: {' '.join(cmd)}")
    return completed


def _check_binary(name: str) -> None:
    if shutil.which(name) is None:
        raise RuntimeError(f"Missing dependency: {name}")


def _write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def _setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("dichvideo_batch_omnivoice_from_srt")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    return logger


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, torch, shutil
from pathlib import Path

MODEL = 'k2-fsa/OmniVoice'
TIMING_MODE = 'no_cut_sequential'  # Kept for compatibility; merged mode creates one audio per SRT.
MAX_TEMPO = '1.35'
NUM_STEP = '32'  # Best quality/stability; use 16 only when you need faster output.
SPEED = '1.0'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float16' if torch.cuda.is_available() else 'float32'
WORKER_PATH = '/content/colab_batch_omnivoice_from_srt.py'
RESULT_ZIP = '/content/omnivoice_audio_results.zip'

cmd = [
    'python', WORKER_PATH,
    '--srt-dir', str(SRT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--ref-audio', REFERENCE_WAV,
    '--ref-text', REF_TEXT,
    '--reference-start', REFERENCE_START,
    '--reference-duration', REFERENCE_DURATION,
    '--model', MODEL,
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--timing-mode', TIMING_MODE,
    '--max-tempo', MAX_TEMPO,
    '--speed', SPEED,
    '--num-step', NUM_STEP,
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

if not Path(RESULT_ZIP).exists():
    print('Zip was not created by worker, creating it from OUTPUT_DIR...')
    shutil.make_archive('/content/omnivoice_audio_results', 'zip', OUTPUT_DIR)

print('Done:', RESULT_ZIP)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

RESULT_ZIP = '/content/omnivoice_audio_results.zip'
if not Path(RESULT_ZIP).exists():
    if Path('/content/dichvideo_omnivoice_audio').exists() and any(Path('/content/dichvideo_omnivoice_audio').rglob('*')):
        shutil.make_archive('/content/omnivoice_audio_results', 'zip', '/content/dichvideo_omnivoice_audio')
    else:
        raise FileNotFoundError('Cannot find /content/omnivoice_audio_results.zip. Run the previous worker cell first and check its error log.')

files.download(RESULT_ZIP)
